- Use Docker

docker run -d --name pgvector \
  -e POSTGRES_USER=tanmay \
  -e POSTGRES_PASSWORD=my_strong_pass \
  -e POSTGRES_DB=rag_project \
  -v pgdata:/var/lib/postgresql/data \
  -p 5432:5432 pgvector/pgvector:pg16

or simply 

docker run -d --name pgvector \
  -e POSTGRES_PASSWORD=postgres -e POSTGRES_DB=ragdb \
  -p 5432:5432 pgvector/pgvector:pg16

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

In [ ]:
loader = PyPDFLoader("attention.pdf")
chunks = loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=30
    )

In [ ]:
docs = splitter.split_documents(chunks)

### Simple Way

#### 1-   For Embedding Store

In [ ]:
import os
from langchain_postgres import PGVector

In [ ]:
CONNECTION = os.environ["DATABASE_URL"]

In [ ]:
vector_store = PGVector(
    embeddings=embeddings,          # same embeddings object you already use
    collection_name="my_docs",      # like your Mongo "namespace"
    connection=CONNECTION,
    use_jsonb=True,                 # store metadata as JSONB
)

In [ ]:
vector_store.add_documents(documents=docs)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

#### 2 - Persistent Data Store

In [ ]:
import uuid
import psycopg
from langchain_postgres import PostgresChatMessageHistory

In [ ]:
conn = psycopg.connect(os.environ["DATABASE_URL"].replace("+psycopg", ""))
table_name = "chat_history"
PostgresChatMessageHistory.create_tables(conn, table_name)

In [ ]:
session_id = str(uuid.uuid4())  # must be a UUID; one per conversation/user
history = PostgresChatMessageHistory(table_name, session_id, sync_connection=conn)

In [ ]:
history.add_user_message("What is RAG?")
history.add_ai_message("RAG combines retrieval with generation...")